# Klasifikasi Gambar Buah & Sayuran menggunakan CNN (Dataset Fruits-360)

**Proyek Akhir — Klasifikasi Gambar (Dicoding)**

> Proyek ini membangun model *Convolutional Neural Network (CNN)* untuk mengklasifikasikan gambar buah & sayuran dari dataset **Fruits-360**. Model dibangun menggunakan **TensorFlow Keras** (Model `Sequential` dengan lapisan `Conv2D`, `BatchNormalization`, `MaxPooling2D`, `Dropout`, `Dense`) lalu diekspor ke tiga format: **SavedModel**, **TF-Lite**, dan **TFJS**.

## Ringkasan Proyek

| Aspek | Keterangan |
|---|---|
| **Topik** | Klasifikasi gambar buah & sayuran (multi-kelas) |
| **Dataset** | Fruits-360 (oleh Mihai Oltean) — Kaggle `moltean/fruits` |
| **Jumlah gambar** | ≥ 90.000 gambar (memenuhi kriteria minimal 1.000) |
| **Jumlah kelas** | ≥ 100 kelas buah & sayuran (memenuhi kriteria minimal 3 kelas) |
| **Resolusi** | Beragam (gambar asli tanpa preprocessing, resolusi tidak seragam) |
| **Split data** | Training (50%), Validation (25%), Test (25%) — sudah disediakan dataset |
| **Arsitektur** | CNN `Sequential`: Conv2D + MaxPooling2D + BatchNorm + Dropout + Dense |
| **Target akurasi** | ≥ 85% pada training & test set (diusahakan ≥ 95%) |

## Alur Pengerjaan

1. Download dataset dari Kaggle
2. Eksplorasi data (jumlah gambar, kelas, resolusi)
3. Split data: train / validation / test
4. Data augmentation (hanya pada data training)
5. Membangun model CNN (Sequential, Conv2D, Pooling)
6. Melatih model dengan callback (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
7. Plot akurasi & loss
8. Evaluasi pada test set
9. Simpan model: SavedModel, TF-Lite, TFJS
10. Inference / uji coba model (TF-Lite)


## 1. Persiapan Lingkungan

> **Penting**: Jalankan notebook ini di **Google Colab** dengan runtime **GPU (T4)** untuk mempercepat pelatihan.
> Menu: *Runtime → Change runtime type → T4 GPU*.


In [1]:
import os
import sys
import random
import json
import math
import shutil
import subprocess

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import Sequential, layers

# Set seed agar hasil dapat direproduksi
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("Python :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))
print("GPU name :", tf.test.gpu_device_name() if tf.test.is_gpu_available(cuda_only=False) else "Tidak ada")


## 2. Download Dataset Fruits-360 dari Kaggle

Dataset **Fruits-360** dapat diunduh dari [Kaggle](https://www.kaggle.com/datasets/moltean/fruits).
Untuk mengunduh via API, siapkan salah satu kredensial Kaggle:

**Cara baru (disarankan) — API Token**
1. Buka [kaggle.com](https://www.kaggle.com) → **Settings → API → Create New Token** (token satu baris, format `KGAT_...`).
2. Jalankan sel di bawah, lalu **tempel token / upload file `access_token`** saat diminta.

**Cara lama — kaggle.json** (username + key): upload file `kaggle.json` saat diminta.

> Catatan keamanan: jangan pernah membagikan token Kaggle Anda ke publik / repo GitHub.


In [1]:
# --- Autentikasi Kaggle (upload kaggle.json) ---
from google.colab import files  # hanya tersedia di Google Colab

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json = os.path.join(kaggle_dir, "kaggle.json")

def save_kaggle_credentials(creds):
    os.environ["KAGGLE_USERNAME"] = creds.get("username", "")
    os.environ["KAGGLE_KEY"] = creds.get("key", "")
    with open(kaggle_json, "w") as f:
        json.dump(creds, f)
    os.chmod(kaggle_json, 0o600)
    print("Kredensial Kaggle tersimpan untuk user:", creds.get("username"))

def save_api_token(token):
    token = token.strip()
    if not token:
        return False
    os.environ["KAGGLE_API_TOKEN"] = token
    with open(os.path.join(kaggle_dir, "access_token"), "w") as f:
        f.write(token)
    os.chmod(os.path.join(kaggle_dir, "access_token"), 0o600)
    return True

def has_old_credentials():
    return bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))

has_token = bool(os.environ.get("KAGGLE_API_TOKEN"))
if has_token:
    save_api_token(os.environ["KAGGLE_API_TOKEN"])
    print("Menggunakan KAGGLE_API_TOKEN dari environment.")
elif has_old_credentials():
    save_kaggle_credentials({
        "username": os.environ["KAGGLE_USERNAME"],
        "key": os.environ["KAGGLE_KEY"],
    })
else:
    print("Silakan upload file kredensial Kaggle pada dialog di bawah:")
    print("  - kaggle.json   (cara lama: username + key), ATAU")
    print("  - access_token  (cara baru: token API format KGAT_...)")
    try:
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        content = open(fname).read().strip()
        try:
            creds = json.loads(content)
            save_kaggle_credentials(creds)
            print("kaggle.json berhasil dimuat untuk user:", creds.get("username"))
        except json.JSONDecodeError:
            if save_api_token(content):
                print("Token API (KGAT_...) berhasil dimuat.")
            else:
                print("Isi file tidak dikenali.")
    except Exception as e:
        print("Tidak ada file yang di-upload:", e)
        token = input("Tempel token API Kaggle Anda (format KGAT_...): ").strip()
        if token.startswith("KGAT_") and save_api_token(token):
            print("Token API berhasil disimpan ke ~/.kaggle/access_token.")
        else:
            print("Token tidak valid. Jalankan ulang sel ini untuk mencoba lagi.")


In [1]:
# --- Download dataset menggunakan kagglehub ---
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
import kagglehub

print("Mengunduh dataset 'moltean/fruits' dari Kaggle ...")
data_root = kagglehub.dataset_download("moltean/fruits")
print("Dataset tersimpan di:", data_root)

def find_split_dirs(root):
    """Temukan semua folder Training / Validation / Test yang berisi folder kelas (folder berisi gambar)."""
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        split = os.path.basename(dirpath).lower()
        if split not in {"training", "validation", "test"}:
            continue
        class_dirs = []
        for d in os.listdir(dirpath):
            p = os.path.join(dirpath, d)
            if os.path.isdir(p):
                n_img = len([f for f in os.listdir(p) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
                if n_img > 0:
                    class_dirs.append(n_img)
        if len(class_dirs) >= 2:
            candidates.append({"path": dirpath, "split": split,
                               "classes": len(class_dirs), "images": sum(class_dirs)})
    groups = {}
    for c in candidates:
        parent = os.path.dirname(c["path"])
        groups.setdefault(parent, {})[c["split"]] = c
    return groups

groups = find_split_dirs(data_root)
print("Versi dataset yang ditemukan:", len(groups))

# Pilih versi: preferensi 'original-size' (resolusi asli tidak seragam), lalu yang punya Validation
best_parent, best_group, best_score = None, None, (-1, -1)
for parent, grp in groups.items():
    has_val = "validation" in grp
    is_original = "original" in os.path.basename(parent).lower()
    total = sum(g["images"] for g in grp.values())
    score = (is_original + has_val, total)
    if score > best_score:
        best_score, best_parent, best_group = score, parent, grp

if best_parent is None:
    raise RuntimeError("Struktur Training/Test tidak ditemukan pada dataset.")

print("Versi dataset terpilih:", os.path.basename(best_parent))

TRAIN_DIR = best_group["training"]["path"]
TEST_DIR = best_group["test"]["path"]
VAL_DIR = best_group.get("validation", {}).get("path")

for label, path in [("TRAIN", TRAIN_DIR), ("VALIDATION", VAL_DIR), ("TEST", TEST_DIR)]:
    if path:
        print(f"  {label}: {path}")


## 3. Eksplorasi Data (EDA)

Melakukan pengecekan jumlah gambar per set, jumlah kelas, variasi resolusi gambar asli, serta visualisasi sampel.


In [1]:
def count_images_in_dir(directory):
    """Hitung total gambar dan jumlah gambar per kelas."""
    total, per_class = 0, {}
    for cls in os.listdir(directory):
        p = os.path.join(directory, cls)
        if not os.path.isdir(p):
            continue
        n = len([f for f in os.listdir(p) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        per_class[cls] = n
        total += n
    return total, per_class

n_train, train_per_class = count_images_in_dir(TRAIN_DIR)
n_test, test_per_class = count_images_in_dir(TEST_DIR)
n_val, val_per_class = (count_images_in_dir(VAL_DIR) if VAL_DIR else (0, {}))

total = n_train + n_val + n_test
print("Jumlah Kelas:", len(train_per_class))
print(f"Training   : {n_train:>7,} gambar ({n_train/total:.1%})")
print(f"Validation : {n_val:>7,} gambar ({n_val/total:.1%})")
print(f"Test       : {n_test:>7,} gambar ({n_test/total:.1%})")
print(f"Total      : {total:>7,} gambar")

def unique_resolutions(directory, max_samples=600):
    """Kumpulkan variasi resolusi gambar asli (subset)."""
    sizes, seen = set(), 0
    for cls in os.listdir(directory):
        p = os.path.join(directory, cls)
        if not os.path.isdir(p):
            continue
        for f in os.listdir(p):
            if not f.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            try:
                with Image.open(os.path.join(p, f)) as im:
                    sizes.add(im.size)
            except Exception:
                pass
            seen += 1
            if seen >= max_samples:
                return sizes
    return sizes

sizes = unique_resolutions(TRAIN_DIR)
print("\nContoh variasi resolusi gambar asli (subset 600 gambar):", sorted(sizes)[:8], "...")
print("Jumlah varian resolusi unik:", len(sizes))
print("\u2705 Resolusi gambar TIDAK seragam (dataset asli tanpa preprocessing)" if len(sizes) > 1
      else "Resolusi gambar seragam.")


In [1]:
# --- Distribusi jumlah gambar per kelas (30 kelas teratas) ---
sorted_classes = sorted(train_per_class.items(), key=lambda x: x[1], reverse=True)[:30]
names = [c[0] for c in sorted_classes]
counts = [c[1] for c in sorted_classes]

plt.figure(figsize=(16, 6))
plt.bar(range(len(counts)), counts, color="#4C72B0")
plt.xticks(range(len(names)), names, rotation=90, fontsize=9)
plt.xlabel("Kelas")
plt.ylabel("Jumlah Gambar")
plt.title("Distribusi Jumlah Gambar per Kelas (30 kelas teratas)")
plt.tight_layout()
plt.show()


In [1]:
# --- Contoh gambar dari beberapa kelas (resolusi asli) ---
classes_sample = sorted(train_per_class.keys())[:4]
fig, axes = plt.subplots(4, 4, figsize=(9, 9))
for row, cls in enumerate(classes_sample):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    files = [f for f in sorted(os.listdir(cls_dir)) if f.lower().endswith((".jpg", ".jpeg", ".png"))][:4]
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(cls_dir, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(cls, fontsize=9, loc="left")
plt.suptitle("Contoh Gambar Dataset (resolusi asli)", fontsize=13)
plt.tight_layout()
plt.show()


## 4. Split Data & Preprocessing (Train / Validation / Test)

- Dataset Fruits-360 sudah menyediakan folder `Training`, `Validation`, dan `Test` dengan proporsi **50% / 25% / 25%**.
- **Augmentasi hanya diterapkan pada data training** agar model lebih general dan tidak overfit.
- Data validation & test hanya di-*rescale* (tanpa augmentasi) demi menjaga konsistensi evaluasi.


In [1]:
IMG_SIZE = 128
BATCH_SIZE = 64

import tensorflow as tf

# Membuat dataset menggunakan image_dataset_from_directory (sangat cepat & modern)
if VAL_DIR:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')
    val_ds = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR, seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=0.2, subset="training", seed=42, 
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=0.2, subset="validation", seed=42, 
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False)

class_names = train_ds.class_names
num_classes = len(class_names)

# Optimasi I/O agar GPU tidak menunggu (bottleneck)
AUTOTUNE = tf.data.AUTOTUNE
train_generator = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_generator = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_generator = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("\nJumlah kelas :", num_classes)
print("3 label pertama :", class_names[:3])


## 5. Arsitektur Model CNN

Model dibangun dengan **`Sequential`** dan lapisan **`Conv2D`** + **`MaxPooling2D`** (pooling layer) + `BatchNormalization` + `Dropout` + `Dense` (softmax).

- Input: gambar ukuran `128 x 128 x 3` (RGB)
- Output: `num_classes` (probabilitas tiap kelas)
- Optimizer: `Adam` dengan `categorical_crossentropy` dan metrik `accuracy`


In [1]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.15, 0.15)
], name="data_augmentation")

model = Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    # Block 1
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Block 4
    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Fully Connected
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 6. Callback & Pelatihan Model

Callback yang digunakan:
- **`EarlyStopping`** — menghentikan pelatihan bila `val_loss` tidak membaik (mencegah overfit).
- **`ReduceLROnPlateau`** — menurunkan *learning rate* bila loss stagnan.
- **`ModelCheckpoint`** — menyimpan model terbaik (`best_model.h5`).


In [1]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint("best_model.h5", monitor="val_accuracy", save_best_only=True, verbose=1),
]

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=25,
    callbacks=callbacks,
    verbose=1,
)


## 7. Plot Akurasi & Loss

Visualisasi performa model selama pelatihan (training vs validation).


In [1]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, acc, "b-", label="Training Accuracy")
axes[0].plot(epochs_range, val_acc, "r-", label="Validation Accuracy")
axes[0].set_title("Akurasi per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, loss, "b-", label="Training Loss")
axes[1].plot(epochs_range, val_loss, "r-", label="Validation Loss")
axes[1].set_title("Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Akurasi training akhir   : {acc[-1]:.4f} ({acc[-1]*100:.2f}%)")
print(f"Akurasi validasi akhir   : {val_acc[-1]:.4f} ({val_acc[-1]*100:.2f}%)")
print(f"Loss training akhir      : {loss[-1]:.4f}")
print(f"Loss validasi akhir      : {val_loss[-1]:.4f}")


## 8. Evaluasi Model pada Test Set

Model dievaluasi terhadap **test set** yang belum pernah dilihat selama pelatihan.


In [1]:
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print("=" * 50)
print(f"Loss pada Test Set    : {test_loss:.4f}")
print(f"Akurasi pada Test Set : {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 50)

# Top-1 dan Top-3 accuracy pada test set (bukti performa tambahan)
y_true = np.concatenate([y for x, y in test_generator], axis=0)
y_true_idx = np.argmax(y_true, axis=1)

preds = model.predict(test_generator, verbose=0)
top3 = np.argsort(preds, axis=1)[:, ::-1][:, :3]
top1_acc = np.mean(top3[:, 0] == y_true_idx)
top3_acc = np.mean([y in row for y, row in zip(y_true_idx, top3)])
print(f"Top-1 Accuracy : {top1_acc:.4f}")
print(f"Top-3 Accuracy : {top3_acc:.4f}")


## 9. Menyimpan Model (SavedModel, TF-Lite, TFJS)

Model disimpan dalam tiga format agar dapat dipakai di berbagai platform:

| Format | Kegunaan |
|---|---|
| **SavedModel** | Deployment di server / cloud (TensorFlow Serving) |
| **TF-Lite** (`model.tflite` + `label.txt`) | Mobile / embedded (Android, iOS, Edge) |
| **TFJS** | Dijalankan di browser / Node.js |


In [1]:
# --- 9a. SavedModel ---
try:
    model.export("saved_model")
    print("SavedModel berhasil diekspor via model.export")
except Exception as e:
    print("model.export gagal, fallback ke tf.saved_model.save:", e)
    tf.saved_model.save(model, "saved_model")
print("Isi folder saved_model:", sorted(os.listdir("saved_model")))

# --- Simpan juga dalam format .keras & .h5 (untuk konversi TFJS) ---
model.save("model.keras")
try:
    model.save("model.h5")
except Exception as e:
    print("Catatan: model.h5 tidak disimpan:", e)


In [1]:
# --- 9b. TF-Lite ---
os.makedirs("tflite", exist_ok=True)
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_bytes = converter.convert()
except Exception as e:
    print("Konversi dari model gagal, fallback dari SavedModel:", e)
    converter = tf.lite.TFLiteConverter.from_saved_model("saved_model")
    tflite_bytes = converter.convert()

with open("tflite/model.tflite", "wb") as f:
    f.write(tflite_bytes)

# label.txt berisi nama kelas sesuai urutan index model
with open("tflite/label.txt", "w") as f:
    f.write("\n".join(class_names))

print("tflite/model.tflite :", round(os.path.getsize("tflite/model.tflite") / 1e6, 2), "MB")
print("tflite/label.txt    :", len(class_names), "label")
print("Label pertama:", class_names[0], "| Label terakhir:", class_names[-1])


In [1]:
# --- 9c. TFJS ---
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflowjs"])
import tensorflowjs as tfjs

try:
    tfjs.converters.save_keras_model(model, "tfjs_model")
    print("TFJS berhasil dikonversi via save_keras_model")
except Exception as e:
    print("save_keras_model gagal, fallback ke CLI tensorflowjs_converter:", e)
    result = subprocess.run(
        ["tensorflowjs_converter", "--input_format=keras_model", "model.keras", "tfjs_model"],
        capture_output=True, text=True)
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])

print("\n--- Struktur hasil konversi ---")
for folder in ["saved_model", "tflite", "tfjs_model"]:
    print(f"\n[{folder}]")
    for root, dirs, files in os.walk(folder):
        for fn in sorted(files):
            rel = os.path.relpath(os.path.join(root, fn), folder)
            size = os.path.getsize(os.path.join(root, fn))
            print(f"  {rel} ({size/1e3:.1f} KB)")


## 10. Inference Menggunakan Model TF-Lite

Membuktikan bahwa model **TF-Lite** (`model.tflite`) dapat digunakan untuk melakukan prediksi pada gambar baru.


In [1]:
interpreter = tf.lite.Interpreter(model_path="tflite/model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input shape :", input_details[0]["shape"])
print("Output shape:", output_details[0]["shape"])

def preprocess_image(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)

# Pilih beberapa gambar dari berbagai kelas pada test set
test_classes = sorted(os.listdir(TEST_DIR))
step = max(1, len(test_classes) // 6)
samples = []
for cls in test_classes[::step][:6]:
    cls_dir = os.path.join(TEST_DIR, cls)
    files = [f for f in sorted(os.listdir(cls_dir)) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if files:
        samples.append((cls, os.path.join(cls_dir, files[0])))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (true_cls, path) in zip(axes.flat, samples):
    interpreter.set_tensor(input_details[0]["index"], preprocess_image(path))
    interpreter.invoke()
    probs = interpreter.get_tensor(output_details[0]["index"])[0]
    pred_idx = int(np.argmax(probs))
    pred_cls = class_names[pred_idx]
    conf = probs[pred_idx]

    img = Image.open(path)
    ax.imshow(img)
    color = "green" if pred_cls == true_cls else "red"
    ax.set_title(f"Benar : {true_cls}\nPrediksi : {pred_cls} ({conf*100:.1f}%)",
                 fontsize=9, color=color)
    ax.axis("off")

plt.suptitle("Inference Menggunakan Model TF-Lite", fontsize=14)
plt.tight_layout()
plt.show()
print("\u2705 Inference TF-Lite berhasil dilakukan (bukti terlampir pada gambar di atas).")


## 11. Simpulan

Model CNN (Sequential + Conv2D + Pooling) berhasil dilatih pada dataset **Fruits-360** dengan:
- **Akurasi training ≥ 85%** (target minimal) dan **akurasi test ≥ 85%** ✔
- Plot akurasi & loss untuk memantau overfitting ✔
- Model disimpan dalam format **SavedModel**, **TF-Lite**, dan **TFJS** ✔
- Inference berhasil menggunakan model TF-Lite ✔


## 12. Kemas Submission & Download

Menyusun folder submission sesuai struktur yang disarankan, lalu mengunduhnya sebagai file `.zip`.

```
submission
├─── tfjs_model
│   ├─── group1-shard1of1.bin
│   └─── model.json
├─── tflite
│   ├─── model.tflite
│   └─── label.txt
├─── saved_model
│   ├─── saved_model.pb
│   └─── variables
├─── notebook.ipynb
├─── README.md
└─── requirements.txt
```


In [1]:
# --- Siapkan folder submission ---
os.makedirs("submission", exist_ok=True)
shutil.copytree("saved_model", "submission/saved_model", dirs_exist_ok=True)
shutil.copytree("tflite", "submission/tflite", dirs_exist_ok=True)
shutil.copytree("tfjs_model", "submission/tfjs_model", dirs_exist_ok=True)

# Salin notebook ini menjadi notebook.ipynb
nb_files = [f for f in os.listdir(".") if f.endswith(".ipynb")]
if nb_files:
    shutil.copy(nb_files[0], "submission/notebook.ipynb")
    print("notebook.ipynb disalin dari:", nb_files[0])

# Buat struktur folder sederhana untuk README
submission_tree = """submission/
├─── tfjs_model/        (model.json, group1-shard1of1.bin)
├─── tflite/            (model.tflite, label.txt)
├─── saved_model/        (saved_model.pb, variables/)
├─── notebook.ipynb
├─── README.md
└─── requirements.txt"""

# --- README.md (dibuat otomatis dengan hasil aktual) ---
readme = f"""# Klasifikasi Gambar Buah & Sayuran dengan CNN (Fruits-360)

## Ringkasan
Proyek klasifikasi gambar menggunakan **Convolutional Neural Network (CNN)** berbasis TensorFlow Keras.
Model dilatih pada dataset **Fruits-360** (\u2265 90.000 gambar, {num_classes} kelas buah & sayuran,
resolusi asli tidak seragam) dan dihasilkan dalam 3 format: **SavedModel**, **TF-Lite**, dan **TFJS**.

## Dataset
- Sumber: Kaggle — [moltean/fruits](https://www.kaggle.com/datasets/moltean/fruits)
- Jumlah gambar: {total:,} (Training {n_train:,} / Validation {n_val:,} / Test {n_test:,})
- Jumlah kelas: {num_classes}
- Resolusi: beragam (gambar asli tanpa preprocessing)

## Hasil Pelatihan
| Metrik | Nilai |
|---|---|
| Akurasi Training | {acc[-1]*100:.2f}% |
| Akurasi Validation | {val_acc[-1]*100:.2f}% |
| Akurasi Test Set | {test_acc*100:.2f}% |
| Loss Test Set | {test_loss:.4f} |

## Arsitektur Model
Model `Sequential` dengan lapisan `Conv2D`, `BatchNormalization`, `MaxPooling2D`, `Dropout`,
`Flatten`, `Dense` (softmax). Optimizer `Adam`, loss `categorical_crossentropy`.
Callback: `EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`.

## Struktur Submission
{submission_tree}

## Cara Menjalankan
1. Buka [Google Colab](https://colab.research.google.com) dengan runtime GPU T4.
2. Upload `kaggle.json` (Kaggle → Settings → API → Create New Token) saat diminta.
3. Jalankan seluruh sel secara berurutan.
4. Download hasilnya dari bagian akhir notebook.

## Lisensi Dataset
Fruits-360 © Mihai Oltean — lisensi [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
"""

with open("submission/README.md", "w") as f:
    f.write(readme)
print("README.md berhasil dibuat.")

# --- requirements.txt ---
requirements = (
    "tensorflow\n"
    "tensorflowjs\n"
    "kagglehub\n"
    "numpy\n"
    "matplotlib\n"
    "pandas\n"
    "scikit-learn\n"
    "pillow\n"
)
with open("submission/requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt berhasil dibuat.")

# --- Zip submission ---
shutil.make_archive("submission", "zip", "/content", "submission")
zip_size = os.path.getsize("/content/submission.zip") / 1e6
print(f"submission.zip berhasil dibuat ({zip_size:.1f} MB)")

# --- Download ke komputer ---
from google.colab import files
files.download("/content/submission.zip")
